In [1]:
import mysql.connector

def fetch_projects():
    conn = mysql.connector.connect(
        host='localhost',
        user='root',
        password='data@123',
        database='monitor-pa-ml'
    )
    cursor = conn.cursor(dictionary=True)
    query = """
        SELECT data_pa_id, judul_pa, judul_pa_en, desc_pa, platform_aplikasi, teknologi_yg_digunakan
        FROM data_pa
    """
    cursor.execute(query)
    projects = cursor.fetchall()
    cursor.close()
    conn.close()
    return projects


In [2]:
ontology = {
    "platforms": ["Mobile App", "Web App", "Desktop App", "Game/VR", "Embedded System"],
    "programming_languages": ["Python", "Java", "Kotlin", "C++", "JavaScript"],
    "frameworks": {
        "web": ["Django", "Flask", "React", "Vue"],
        "mobile": ["Flutter", "React Native"]
    },
    "databases": ["Firebase", "MySQL", "MongoDB"],
    "tools": ["Figma", "Unity", "Blender", "Visual Studio"]
}


In [3]:
def map_to_ontology(project, ontology):
    # Normalize platform
    platform = (project.get("platform_aplikasi") or "").strip().title()

    mapped_platform = None
    for known_platform in ontology.get("platforms", []):
        if platform == known_platform:
            mapped_platform = known_platform
            break

    # Normalize technologies
    raw_techs = project.get("teknologi_yg_digunakan")
    mapped_techs = []
    if raw_techs:
        import ast
        try:
            tech_list = ast.literal_eval(raw_techs)
            if not isinstance(tech_list, list):
                tech_list = [tech_list]
        except Exception:
            tech_list = [raw_techs]

        for tech in tech_list:
            tech = tech.strip().title()
            found = False

            # Search in flat categories
            for cat in ['programming_languages', 'databases', 'tools']:
                if tech in ontology.get(cat, []):
                    mapped_techs.append((cat, tech))
                    found = True
                    break

            # Search in nested framework categories
            if not found:
                for subcat, items in ontology.get("frameworks", {}).items():
                    if tech in items:
                        mapped_techs.append((f"frameworks:{subcat}", tech))
                        found = True
                        break

            if not found:
                mapped_techs.append(("unknown", tech))

    return mapped_platform, mapped_techs


In [4]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_semantic_vector(judul, desc):
    combined = f"{judul} {desc}"
    return sbert_model.encode(combined)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import ast

def process_all_projects():
    raw_projects = fetch_projects()
    processed = []
    
    for project in raw_projects:
        platform, techs = map_to_ontology(project, ontology)
        vector = get_semantic_vector(project['judul_pa'], project['desc_pa'])
        
        processed.append({
            "id": project['data_pa_id'],
            "judul": project['judul_pa'],
            "desc": project['desc_pa'],
            "platform": platform,
            "techs": techs,
            "vector": vector
        })
    
    return processed


In [6]:
def compute_similarity(project1, project2, weight_text=0.7, weight_tech=0.3):
    # Semantic cosine similarity
    sim_text = cosine_similarity(
        [project1["vector"]], [project2["vector"]]
    )[0][0]

    # Ontology-based tech similarity
    techs1 = set([t[1] for t in project1["techs"]])
    techs2 = set([t[1] for t in project2["techs"]])
    common_techs = techs1.intersection(techs2)
    tech_sim = len(common_techs) / max(len(techs1.union(techs2)), 1)

    # Optional: Platform match (binary)
    platform_sim = 1 if project1["platform"] == project2["platform"] else 0

    total_similarity = (weight_text * sim_text) + (weight_tech * ((tech_sim + platform_sim) / 2))
    return total_similarity


In [7]:
def get_top_similar(projects, target_project, top_k=3):
    similarities = []
    for project in projects:
        if project["id"] == target_project["id"]:
            continue
        sim = compute_similarity(target_project, project)
        similarities.append((project, sim))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

def suggest_innovations(target_project, similar_projects):
    seen_techs = set()
    for proj, _ in similar_projects:
        seen_techs.update([tech for _, tech in proj["techs"]])
    
    target_techs = set([tech for _, tech in target_project["techs"]])
    unique_techs = target_techs - seen_techs
    return list(unique_techs)


In [8]:
projects = process_all_projects()
for proj in projects:
    top_similars = get_top_similar(projects, proj)
    innovations = suggest_innovations(proj, top_similars)
    
    print(f"\n🔍 Project: {proj['judul']}")
    print(f"Most Similar Projects:")
    for sim_proj, score in top_similars:
        print(f"- {sim_proj['judul']} (Score: {score:.2f})")
    
    if innovations:
        print("🚀 Recommend these unique techs/platforms:")
        print(", ".join(innovations))



🔍 Project: YUCOMS: Aplikasi Sistem Kompos Yuki berbasis android pada PT. Yoriyusakifa Yuki Compost
Most Similar Projects:
- MONEYQU: MANAJEMEN KEUANGAN PRIBADI BERABASIS ANDROID (Score: 0.80)
- E-PANGKALAN: APLIKASI MANAJEMEN PANGKALAN GAS DAN AIR (Score: 0.74)
- Aplikasi Kolaborasi Mesjid (Score: 0.74)

🔍 Project: APLIKASI UNTUK MENGELOLA KONTRAKAN BERBASIS ANDROID
Most Similar Projects:
- APLIKASI PENCATATAN NILAI PADA SEKOLAH DASAR BERBASIS ANDROID (Score: 0.81)
- Jago Menabung : Aplikasi Untuk Mengelola Uang Saku Berbasis Mobile (Score: 0.80)
- MyAnimolz : Aplikasi Penitipan Hewan Berbasis Android (Score: 0.78)

🔍 Project: SHAMAN: EDUKASI TENTANG SALAHNYA PESUGIHAN MELALUI GAME HOROR PC
Most Similar Projects:
- Perancangan Aplikasi Permainan "Kemuliaan dan Kehormatan : Perang Legenda", Sebagai Media Promosi Kebudayaan Daerah di Indonesia (Score: 0.80)
- Aplikasi Permainan Bergenre Construction Management Simulation Game Untuk Pembelajaran Sejarah Penyebaran Islam Di Indonesia (Sco

In [9]:
def create_manual_project(judul, desc, platform, raw_techs, ontology):
    project = {
        "judul_pa": judul,
        "desc_pa": desc,
        "platform_aplikasi": platform,
        "teknologi_yg_digunakan": str(raw_techs)  # must be a string like your DB data
    }
    
    # Map platform and techs
    mapped_platform, mapped_techs = map_to_ontology(project, ontology)
    
    # Get semantic vector
    vector = get_semantic_vector(judul, desc)
    
    # Return in the processed project format
    return {
        "id": "manual_input",  # since manual, no DB id
        "judul": judul,
        "desc": desc,
        "platform": mapped_platform,
        "techs": mapped_techs,
        "vector": vector
    }


In [10]:
# First, process all projects from DB
processed_projects = process_all_projects()

manual_proj = create_manual_project(
    judul="Pelacak Kesehatan Berbasis AI",
    desc="Aplikasi mobile yang melacak metrik kesehatan pengguna dan memprediksi risiko penyakit.",
    platform="Mobile App",
    raw_techs="['Python', 'Flutter', 'Firebase']",
    ontology=ontology
)

# Get top similar projects from DB projects to this manual one
top_similar_manual = get_top_similar(processed_projects, manual_proj)

# Suggest innovations unique to manual project
innovations_manual = suggest_innovations(manual_proj, top_similar_manual)

# Print results
print(f"\n🔍 Manual Project: {manual_proj['judul']}")
print(f"Most Similar Projects:")
for sim_proj, score in top_similar_manual:
    print(f"- {sim_proj['judul']} (Score: {score:.2f})")

if innovations_manual:
    print("🚀 Recommend these unique techs/platforms:")
    print(", ".join(innovations_manual))
else:
    print("No unique innovations found compared to similar projects.")



🔍 Manual Project: Pelacak Kesehatan Berbasis AI
Most Similar Projects:
- Mengger : Aplikasi Pengelolaan Keuangan Dengan Fitur Pantau Berbasis Android (Score: 0.75)
- MyTelU: Aplikasi mobile untuk civitas academia Telkom University berbasis Flutter (Score: 0.75)
- Jago Menabung : Aplikasi Untuk Mengelola Uang Saku Berbasis Mobile (Score: 0.74)
🚀 Recommend these unique techs/platforms:
Python


In [11]:
print(manual_proj.keys())
print(processed_projects[0].keys())


dict_keys(['id', 'judul', 'desc', 'platform', 'techs', 'vector'])
dict_keys(['id', 'judul', 'desc', 'platform', 'techs', 'vector'])


In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def semantic_similarity(desc1, desc2):
    desc1 = desc1 or ""
    desc2 = desc2 or ""
    if desc1 == "" or desc2 == "":
        return 0.0
    emb1 = model.encode(desc1)
    emb2 = model.encode(desc2)
    cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return cosine_sim


def combined_similarity(proj1, proj2, alpha=0.7):
    sem_sim = semantic_similarity(proj1['desc'], proj2['desc'])
    
    tech_set1 = set(proj1.get('techs', []))
    tech_set2 = set(proj2.get('techs', []))
    
    if tech_set1.union(tech_set2):
        tech_sim = len(tech_set1.intersection(tech_set2)) / len(tech_set1.union(tech_set2))
    else:
        tech_sim = 0
    
    platform1 = proj1.get('platform') or ''
    platform2 = proj2.get('platform') or ''
    
    platform_sim = 1 if platform1.lower() == platform2.lower() else 0
    
    combined_score = alpha * sem_sim + (1 - alpha) * ((tech_sim + platform_sim) / 2)
    return combined_score



# Example usage:
for db_proj in processed_projects:
    score = combined_similarity(manual_proj, db_proj)
    print(f"Similarity with '{db_proj['judul']}': {score:.3f}")


Similarity with 'YUCOMS: Aplikasi Sistem Kompos Yuki berbasis android pada PT. Yoriyusakifa Yuki Compost': 0.384
Similarity with 'APLIKASI UNTUK MENGELOLA KONTRAKAN BERBASIS ANDROID': 0.479
Similarity with 'SHAMAN: EDUKASI TENTANG SALAHNYA PESUGIHAN MELALUI GAME HOROR PC': 0.178
Similarity with 'Pengembangan Aplikasi Web Konten Pembelajaran Digital Studycle Kids': 0.286
Similarity with '“Akibat”: Game Horor Perdukunan  Berbasis Mobile': 0.077
Similarity with 'APLIKASI WEB UNTUK MELAKUKAN PEMANTAUAN PROYEK AKHIR MAHASISWA': 0.265
Similarity with 'Aplikasi Mengenal Huruf Hijaiyah': 0.160
Similarity with 'Mussia AR : Aplikasi Interaktif Museum Pos Indonesia Berbasis Augmented Reality': 0.306
Similarity with 'Mengger : Aplikasi Pengelolaan Keuangan Dengan Fitur Pantau Berbasis Android': 0.520
Similarity with 'Easy English : Game Belajar Bahasa Inggris Berbasis Android Menggunakan Metode Phonic': 0.104
Similarity with 'Aplikasi Kosakata Quran Berbasis Android  sebagai Sarana dalam Memahami 

In [13]:
# Get similarity scores between manual project and all processed DB projects
similarity_scores = []
for db_proj in processed_projects:
    score = combined_similarity(manual_proj, db_proj)
    similarity_scores.append((db_proj, score))

# Sort by similarity score in descending order
similarity_scores.sort(key=lambda x: x[1], reverse=True)

# Get top 10 similar projects
top_10_similar = similarity_scores[:10]

# Display results
print(f"\n🔍 Manual Project: {manual_proj['judul']}")
print("Top 10 Similar Projects:")
for i, (proj, sim_score) in enumerate(top_10_similar, 1):
    print(f"{i}. {proj['judul']} — Similarity Score: {sim_score:.3f}")



🔍 Manual Project: Pelacak Kesehatan Berbasis AI
Top 10 Similar Projects:
1. Checkbund: Aplikasi Monitoring Kesehatan Ibu Hamil Berbasis Android — Similarity Score: 0.655
2. Healthify : Aplikasi Diet dan Pola Hidup Sehat Berbasis Android — Similarity Score: 0.647
3. Healthify : Aplikasi Diet dan Pola Hidup Sehat Berbasis Android — Similarity Score: 0.647
4. PEMBANGUNAN APLIKASI MOBILE UNTUK LAYANAN HOME CARE KESEHATAN DALAM MENINGKATKAN INTERAKSI ANTARA MASYARAKAT DAN TENAGA MEDIS — Similarity Score: 0.622
5. iBun: Aplikasi Notifikasi Kesehatan Selama Masa Kehamilan Berbasis Android — Similarity Score: 0.619
6. Jago Menabung : Aplikasi Untuk Mengelola Uang Saku Berbasis Mobile — Similarity Score: 0.610
7. Aplikasi Android Pencarian Ruang Kos Berbasis Lokasi — Similarity Score: 0.596
8. APLIKASI MOBILE BERBASIS ANDROID UNTUK PEMESANAN DAN PENGELOLAAN LAYANAN SERVICE KENDARAAN — Similarity Score: 0.586
9. Aplikasi Pencatatan Kesehatan Ibu Hamil dan Balita — Similarity Score: 0.580
10. My